# 04 — Ghép fact và huấn luyện Spark LR/GBT

### 1. Dựng fact từ Silver và dimension

Chạy sau notebook 03. `build_fact()` ghép mỗi đơn Silver với phiên bản
shipper theo ngày nhận và các khóa vùng, AOI, ngày nhận, ngày giao, giờ nhận.
Hàm ghi `fact_delivery_task_gold` và đối soát khóa với Silver.

In [ ]:
import os

from datetime import datetime

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.sql.window import Window

from lakehouse.common import attach_dimension_keys, haversine, join_courier_history, parse_timestamp
from lakehouse.notebooks import load_notebook_steps

shared = load_notebook_steps()
DIM = shared.DIM
DIMS = shared.DIMS
FACT = shared.FACT
BRONZE = shared.BRONZE
PRED = shared.PRED
RUN_INFO = shared.RUN_INFO
SILVER = shared.SILVER
reconcile_order_keys = shared.reconcile_order_keys
write_mysql = shared.write_mysql

spark = shared.get_spark("Delivery - Fact and ML Gold")

In [ ]:
def build_fact(spark):
    silver = spark.read.format("delta").load(SILVER)
    dim = spark.read.format("delta").load(DIM)
    fact = join_courier_history(silver, dim)
    fact = attach_dimension_keys(fact, {n: spark.read.format("delta").load(p) for n, p in DIMS.items()})
    fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FACT)
    fact = spark.read.format("delta").load(FACT)
    reconcile_order_keys(silver, fact, "Silver → Fact")
    return fact

In [ ]:
fact = build_fact(spark)

In [2]:
fact.select(
    "order_id", "courier_sk", "region_sk", "aoi_sk", "accept_date_key",
    "delivery_date_key", "accept_hour_key", "delivery_minutes",
).show(5, truncate=False)
print("Số dòng fact:", fact.count())
print("order_id duy nhất:", fact.select("order_id").distinct().count())

+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|order_id|courier_sk          |region_sk|aoi_sk|accept_date_key|delivery_date_key|accept_hour_key|delivery_minutes|
+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|2071340 |-5931354023872468343|1        |14    |20220618       |20220618         |8              |501.0           |
|3767291 |-5931354023872468343|1        |14    |20220814       |20220814         |8              |391.0           |
|2184812 |-5931354023872468343|1        |14    |20220901       |20220901         |9              |367.0           |
|4048066 |-6655371896403481998|1        |28    |20220825       |20220825         |13             |267.0           |
|2309133 |-306575199876071777 |1        |29    |20220709       |20220709         |8              |119.0           |
+--------+--------------------+---------+------+---------------+--------

### 2. Dựng đặc trưng từ các sự kiện đã quan sát

Bronze cung cấp sự kiện nhận đơn và giao xong. Điểm truy vấn là lúc nhận.
Các đặc trưng về shipper đọc sự kiện có timestamp nhỏ hơn mốc này. Nhiều
đơn nhận cùng timestamp nhận cùng thứ hạng và cùng lịch sử.

GPS phải có timestamp tới thời điểm nhận và tọa độ hợp lệ. GPS thiếu
được đưa vào cột cờ. Tập fit học giá trị điền thiếu. Fact cung cấp nhãn và
khóa báo cáo. Giữ cả đơn thời lượng dài và đơn thiếu GPS khi chấm.

In [ ]:
SPARK_BASELINE_FEATURES = [
    "gps_distance_km", "gps_missing", "accept_hour", "dow", "order_rank",
    "courier_prior_distance_km", "courier_completed_avg_min", "courier_completed_count",
]
SPARK_TEST_ALL = "test (tất cả)"
SPARK_TEST_SHORT = "test (<=24h)"


def spark_baseline_features(bronze):
    """Build Spark covariates from the accept and completion event stream."""
    events = bronze.select(
        "order_id", F.col("courier_id").cast("long").alias("courier_id"),
        parse_timestamp("accept_time").alias("accept_ts"),
        parse_timestamp("delivery_time").alias("delivery_ts"),
        parse_timestamp("accept_gps_time").alias("accept_gps_ts"),
        *(F.col(name).cast("double").alias(name)
          for name in ["lat", "lng", "accept_gps_lat", "accept_gps_lng"]),
    ).filter(F.col("accept_ts").isNotNull())
    distance = F.when(
        F.col("accept_gps_ts") <= F.col("accept_ts"),
        haversine("lat", "lng", "accept_gps_lat", "accept_gps_lng"),
    )
    accepted = events.select(
        "order_id", "courier_id", F.col("accept_ts").alias("event_ts"),
        F.lit(1).alias("is_accept"), distance.alias("gps_distance_km"),
        F.lit(None).cast("double").alias("completed_minutes"),
    )
    completed = events.filter(F.col("delivery_ts") > F.col("accept_ts")).select(
        "order_id", "courier_id", F.col("delivery_ts").alias("event_ts"),
        F.lit(0).alias("is_accept"), F.lit(None).cast("double").alias("gps_distance_km"),
        ((F.unix_micros("delivery_ts") - F.unix_micros("accept_ts")) / 60_000_000).alias("completed_minutes"),
    )
    timeline = (accepted.unionByName(completed).withColumn("event_day", F.to_date("event_ts"))
                .withColumn("event_us", F.unix_micros("event_ts")))
    earlier = (Window.partitionBy("courier_id").orderBy("event_us")
               .rangeBetween(Window.unboundedPreceding, -1))
    earlier_today = (Window.partitionBy("courier_id", "event_day").orderBy("event_us")
                     .rangeBetween(Window.unboundedPreceding, -1))
    has_courier = F.col("courier_id").isNotNull()
    return (timeline
            .withColumn("order_rank", 1 + F.coalesce(F.when(has_courier, F.sum("is_accept").over(earlier_today)), F.lit(0)))
            .withColumn("courier_prior_distance_km", F.when(has_courier, F.avg("gps_distance_km").over(earlier_today)))
            .withColumn("courier_completed_avg_min", F.when(has_courier, F.avg("completed_minutes").over(earlier)))
            .withColumn("courier_completed_count", F.coalesce(F.when(has_courier, F.count("completed_minutes").over(earlier)), F.lit(0)))
            .filter(F.col("is_accept") == 1)
            .withColumn("gps_missing", F.col("gps_distance_km").isNull().cast("double"))
            .withColumn("accept_hour", F.hour("event_ts") + F.minute("event_ts") / 60)
            .withColumn("dow", F.dayofweek("event_day"))
            .select("order_id", *SPARK_BASELINE_FEATURES))


def split_spark_baseline_rows(rows, train_end):
    """Freeze model labels before the date used to start scoring."""
    cutoff = F.lit(train_end).cast("timestamp")
    fit = rows.filter((F.col("accept_ts") < cutoff) & (F.col("delivery_ts") < cutoff))
    scored = rows.filter(F.col("accept_ts") >= cutoff)
    return fit, scored

In [ ]:
bronze = spark.read.format("delta").load(BRONZE)
state = spark_baseline_features(bronze)
fit_orders, score_orders = split_spark_baseline_rows(fact, "2022-10-01")
print("Đơn fit đã biết nhãn", fit_orders.count())
print("Đơn chấm thuộc Fact từ ngày cắt", score_orders.count())
print("Lát chấm ≤1.440 phút", score_orders.filter(F.col("delivery_minutes") <= 1440).count())
state.select("order_id", "order_rank", "courier_completed_count", "courier_completed_avg_min",
             "gps_distance_km", "gps_missing").show(5, truncate=False)

### 3. Chia theo ngày và fit hai baseline

Với ngày cắt mặc định 01/10/2022, tập fit gồm đơn nhận và đã giao trước
ngày đó. Test gồm mọi đơn có nhãn hợp lệ trong Fact nhận từ ngày cắt
trở đi. Cả LR và GBT học trên số phút. GBT dùng absolute loss và số cây
cố định. Báo cáo bổ sung lát ≤1.440 phút sau khi đã dự đoán.

Quy tắc là nếu shipper đã nhận ba đơn sớm hơn trong ngày thì hạng nhận
tiếp theo là 4. Nếu trước lần nhận này mới có hai đơn giao xong với thời
lượng 32 và 50 phút, đặc trưng trung bình shipper là 41 phút.

Mã hóa vùng, AOI và các median điền thiếu được fit trên tập train.
train_spark_baseline() ghi eta_predictions_gold và quản lý cache.
Prefect gọi build_fact_and_train() để dựng fact rồi fit trong cùng lượt.
[Benchmark Spark trên cùng mẫu Protocol B](../docs/eta/training_data_lineage.md#đối-chiếu-lightgbm-và-spark)

In [ ]:
def _metrics(df, label, pred_col):
    error = F.col(pred_col) - F.col(label)
    row = df.agg(
        F.count("*").alias("n"), F.avg(F.abs(error)).alias("mae"),
        F.avg(error ** 2).alias("mse"), F.var_pop(label).alias("label_variance"),
    ).first()
    if row.n == 0:
        return 0, None, None, None
    r2 = 1 - row.mse / row.label_variance if row.label_variance else None
    return row.n, float(row.mae), float(row.mse ** 0.5), r2

In [ ]:
def _train_baseline(fact, train_end, late_threshold_min, run_id, cached):
    from eta.model import SPARK_GBT_BASELINE

    spark = fact.sparkSession
    history = spark_baseline_features(spark.read.format("delta").load(BRONZE))
    ml = (fact.select("order_id", "courier_id", "region_id", "aoi_type", "order_date",
                      "accept_ts", "delivery_ts", "delivery_minutes")
          .join(history, "order_id", "left")
          .withColumn("distance_km", F.col("gps_distance_km"))
          .withColumn("aoi_type_str", F.col("aoi_type").cast("string"))
          .withColumn("region_str", F.col("region_id").cast("string"))
          .cache())
    cached.append(ml)
    if ml.filter(F.col("order_rank").isNull()).limit(1).count():
        raise ValueError("Đơn trong Fact thiếu sự kiện nhận trong Bronze")
    n_silver = ml.count()
    train, test = split_spark_baseline_rows(ml, train_end)
    train, test = train.cache(), test.cache()
    cached.extend([train, test])
    n_train, n_test = train.count(), test.count()
    if n_train < 2 or n_test < 2:
        raise ValueError("Tập fit có nhãn kịp thời hoặc tập chấm quá nhỏ, kiểm tra train_end")

    quantiles = train.approxQuantile(SPARK_BASELINE_FEATURES, [0.5], 0.01)
    defaults = {name: float(values[0]) if values else 0.0
                for name, values in zip(SPARK_BASELINE_FEATURES, quantiles)}
    train_inputs, test_inputs = train.fillna(defaults), test.fillna(defaults)
    idx = Pipeline(stages=[
        StringIndexer(inputCol="aoi_type_str", outputCol="aoi_idx", handleInvalid="keep"),
        StringIndexer(inputCol="region_str", outputCol="region_idx", handleInvalid="keep"),
    ]).fit(train_inputs)
    tr, te = idx.transform(train_inputs), idx.transform(test_inputs)
    ohe = OneHotEncoder(inputCols=["aoi_idx", "region_idx"], outputCols=["aoi_ohe", "region_ohe"],
                        handleInvalid="keep").fit(tr)
    tr, te = ohe.transform(tr), ohe.transform(te)
    a_lr = VectorAssembler(inputCols=SPARK_BASELINE_FEATURES + ["aoi_ohe", "region_ohe"], outputCol="features_lr")
    a_gbt = VectorAssembler(inputCols=SPARK_BASELINE_FEATURES + ["aoi_idx", "region_idx"], outputCol="features_gbt")
    tr = a_gbt.transform(a_lr.transform(tr)).cache()
    cached.append(tr)
    te = a_gbt.transform(a_lr.transform(te))

    lr = LinearRegression(featuresCol="features_lr", labelCol="delivery_minutes", predictionCol="p_lr").fit(tr)
    max_iter, step_size = SPARK_GBT_BASELINE
    gbt = GBTRegressor(
        featuresCol="features_gbt", labelCol="delivery_minutes", predictionCol="p_gbt",
        lossType="absolute", maxIter=max_iter, stepSize=step_size, maxDepth=5, maxBins=256, seed=42,
    ).fit(tr)

    med = train.approxQuantile("delivery_minutes", [0.5], 0.01)[0]
    mean = train.agg(F.avg("delivery_minutes")).first()[0]
    pred = (gbt.transform(lr.transform(te))
            .withColumn("pred_lr", F.greatest(F.col("p_lr"), F.lit(0.0)))
            .withColumn("pred_gbt", F.greatest(F.col("p_gbt"), F.lit(0.0)))
            .withColumn("pred_median", F.lit(float(med)))
            .withColumn("pred_mean", F.lit(float(mean)))
            .select("order_id", "courier_id", "region_id", "aoi_type", "order_date", "distance_km",
                    "accept_hour", "delivery_minutes", "pred_lr", "pred_gbt", "pred_median", "pred_mean")
            .cache())
    cached.append(pred)
    pred.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(PRED)

    run_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []
    model_columns = [
        ("Baseline: trung vị", "pred_median"), ("Baseline: trung bình", "pred_mean"),
        ("Linear Regression", "pred_lr"), ("GBT", "pred_gbt"),
    ]
    for cohort, frame in [(SPARK_TEST_ALL, pred), (SPARK_TEST_SHORT, pred.filter(F.col("delivery_minutes") <= 1440))]:
        for name, col in model_columns:
            n, mae, rmse, r2 = _metrics(frame, "delivery_minutes", col)
            if not n:
                break
            rows.append((name, cohort, n, round(mae, 2), round(rmse, 2),
                         round(r2, 4) if r2 is not None else None, run_ts))
    b_mae, b_rmse = os.getenv("BENCHMARK_MAE"), os.getenv("BENCHMARK_RMSE")
    if b_mae and b_rmse:
        rows.append((os.getenv("BENCHMARK_NAME", "Benchmark bài báo gốc"), "theo bài báo",
                     None, float(b_mae), float(b_rmse), None, run_ts))
    metrics = spark.createDataFrame(
        rows, "model string, cohort string, n long, mae double, rmse double, r2 double, run_ts string")
    write_mysql(metrics, "ml_metrics", run_id=run_id)

    names = SPARK_BASELINE_FEATURES + ["aoi_type", "region_id"]
    imp = spark.createDataFrame(
        [(n_, float(v)) for n_, v in zip(names, gbt.featureImportances.toArray())],
        "feature string, importance double")
    write_mysql(imp, "ml_feature_importance", run_id=run_id)

    if late_threshold_min is None:
        late_threshold_min = float(train.approxQuantile("delivery_minutes", [0.9], 0.01)[0])
    unavailable_labels = n_silver - n_train - n_test
    info = spark.createDataFrame(
        [(run_ts, str(train_end), n_silver, n_train, n_test, float(late_threshold_min),
          unavailable_labels, gbt.getNumTrees, step_size, "absolute")],
        "run_ts string, train_end string, silver_rows long, train_rows long, test_rows long, "
        "late_threshold_min double, unavailable_train_labels long, gbt_trees int, gbt_step_size double, gbt_loss string")
    info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(RUN_INFO)
    write_mysql(info, "pipeline_run_info", run_id=run_id)
    out = {r[0]: {"mae": r[3], "rmse": r[4], "r2": r[5]} for r in rows if r[1] == SPARK_TEST_ALL}
    out.update({"train_rows": n_train, "test_rows": n_test, "unavailable_train_labels": unavailable_labels,
                "gbt_trees": gbt.getNumTrees, "late_threshold_min": late_threshold_min})
    return out

In [ ]:
def train_spark_baseline(fact, train_end="2022-10-01", late_threshold_min=None, *, run_id=None):
    cached = []
    try:
        return _train_baseline(fact, train_end, late_threshold_min, run_id, cached)
    finally:
        for frame in reversed(cached):
            frame.unpersist()


def build_fact_and_train(spark, train_end="2022-10-01", late_threshold_min=None, *, run_id=None):
    return train_spark_baseline(build_fact(spark), train_end, late_threshold_min, run_id=run_id)

In [ ]:
result = train_spark_baseline(fact, train_end="2022-10-01")
result

<details>
<summary>Kết quả lịch sử của baseline Spark trước sửa leakage</summary>

Bảng này thuộc cohort GPS, 50 km và thời lượng tại lần chạy cũ.

~~~text
{'Baseline: trung vị': {'mae': 127.59, 'rmse': 169.45, 'r2': -0.0494},
 'Baseline: trung bình': {'mae': 131.0, 'rmse': 165.42, 'r2': -0.0001},
 'Linear Regression': {'mae': 109.88, 'rmse': 168.64, 'r2': -0.0394},
 'GBT': {'mae': 92.29, 'rmse': 136.19, 'r2': 0.322},
 'train_rows': 165222,
 'test_rows': 35778,
 'late_threshold_min': 467.0}
~~~

</details>
